##### Using Version

In [ ]:
df = spark.read.format("csv").option("header","true").load("Files/product/products.csv")
display(df)

In [ ]:
%%sql

select * from product

In [ ]:
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.functions import lit

operation_id = ""

source_with_id = df.withColumn("operation_id", lit(operation_id))

deltaTable = DeltaTable.forPath(spark, "Tables/product")
start_version = deltaTable.history().agg({"version": "max"}).collect()[0][0]

nof_records = df.count()

if nof_records == 0:
    print("No records in the file")
else:
    merge_result = deltaTable.alias("target").merge(
        source_with_id.alias("source"),
        "target.productid = source.productid"
    ).whenMatchedUpdate(
        condition="source.is_modified == 1",
        set={
        "name": "source.name",
        "category": "source.category",
        "price": "source.price",
        "operation_id": "source.operation_id"
    }).whenNotMatchedInsert(values={
        "productid": "source.productid",
        "name": "source.name",
        "category": "source.category",
        "subcategory": "source.category",
        "brand": "source.brand",
        "description": "source.description",
        "price": "source.price",
        "color": "source.color",
        "operation_id": "source.operation_id"
    }).execute()
    print("File processed")


In [ ]:

history = deltaTable.history().filter(f"version > {start_version}")
display(history)

In [ ]:
if nof_records > 0:

    if history.count() > 0:
        # Mighgt get multiple****
        latest_commit = history.orderBy("version", ascending=False).limit(1)

        # Collect the commit details
        commit_info = latest_commit.collect()[0]
        operation_metrics = commit_info["operationMetrics"]

        # Print metrics
        print("Records updated")
        print("Number of rows inserted:", operation_metrics.get("numTargetRowsInserted", 0))
        print("Number of rows updated:", operation_metrics.get("numTargetRowsUpdated", 0))
    else:
        print("No operations performed")
        print("Number of rows inserted: 0")
        print("Number of rows updated: 0")
else:
    print("No records to update")

In [ ]:
operation_id